# 5_Ingesta_Bronze_Ventas_v1

## Objetivo
Ingerir el archivo **ventas.csv** desde la capa **Raw** de Azure Data Lake Storage Gen2 hacia la capa **Bronze**, almacenándolo en formato Delta y registrando metadatos técnicos de ingesta.

Se utiliza Managed Identity mediante las External Locations configuradas en Unity Catalog. La lectura inicial permite observar la inferencia de tipos de Spark y la segunda lectura aplica un esquema explícito definido para este dataset.

Este notebook es independiente y forma parte del proceso de ingesta Bronze del proyecto.

## 1. Configuración inicial

In [ ]:
dbutils.widgets.removeAll()

In [ ]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DateType, TimestampType, DecimalType
)

## 2. Parámetros de ejecución

In [ ]:
dbutils.widgets.removeAll()

dbutils.widgets.text("containerRaw", "raw")
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")

containerRaw = dbutils.widgets.get("containerRaw")
containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

esquema = "bronze"

ruta_raw = f"abfss://{containerRaw}@{storageName}.dfs.core.windows.net/"
ruta_bronze = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/bronze/"

print("Catálogo:", catalogo)
print("Esquema:", esquema)
print("Ruta Raw:", ruta_raw)
print("Ruta Bronze:", ruta_bronze)

## 3. Ingesta de ventas

In [ ]:
# Lectura inicial para observar los tipos inferidos por Spark
df_ventas_inferido = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv(f"{ruta_raw}ventas.csv")

df_ventas_inferido.printSchema()

# Definición manual del esquema
ventas_schema = StructType([
    StructField("venta_id", IntegerType(), True),
    StructField("fecha_venta", TimestampType(), True),
    StructField("cliente_id", IntegerType(), True),
    StructField("producto_id", IntegerType(), True),
    StructField("almacen_id", IntegerType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio_unitario", DecimalType(12, 2), True),
    StructField("descuento_pct", DecimalType(5, 2), True),
    StructField("metodo_pago", StringType(), True)
])

# Lectura aplicando el esquema definido
df_ventas = spark.read\
    .option("header", True)\
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")\
        .schema(ventas_schema)\
    .csv(f"{ruta_raw}ventas.csv")

df_ventas = df_ventas\
    .withColumn("_source_file", lit("ventas.csv"))\
    .withColumn("_ingestion_timestamp", current_timestamp())

df_ventas.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("path", f"{ruta_bronze}ventas/")\
    .saveAsTable(f"{catalogo}.{esquema}.ventas")

print("Ventas:", df_ventas.count())

## 4. Validación de la ingesta

In [ ]:
# Validación final de la ingesta
nombre_tabla = f"{catalogo}.{esquema}.ventas"
df_validacion = spark.table(nombre_tabla)

cantidad_obtenida = df_validacion.count()
cantidad_esperada = 100020

print("Tabla validada:", nombre_tabla)
print("Registros esperados:", cantidad_esperada)
print("Registros obtenidos:", cantidad_obtenida)
print("Esquema:")
df_validacion.printSchema()

# Verificación de metadatos técnicos
columnas_requeridas = ['_source_file', '_ingestion_timestamp']
columnas_faltantes = [c for c in columnas_requeridas if c not in df_validacion.columns]

if columnas_faltantes:
    raise ValueError(f"Faltan columnas técnicas: {columnas_faltantes}")

if cantidad_obtenida != cantidad_esperada:
    raise ValueError(
        f"Ingesta no válida para ventas: "
        f"se esperaban {cantidad_esperada} registros y se obtuvieron {cantidad_obtenida}."
    )

print("VALIDACIÓN EXITOSA: la tabla Bronze contiene el total esperado y sus metadatos técnicos.")

## 5. Resultado esperado

| Tabla | Registros esperados |
|---|---:|
| `ventas` | 100,020 |

La tabla Bronze conserva los registros de origen, incluidos los valores nulos y las incidencias detectadas durante el perfilado. La depuración y las reglas de calidad se implementarán en Silver.

La tabla incluye las columnas técnicas `_source_file` y `_ingestion_timestamp`.

**Criterio de aceptación:** la última celda debe finalizar con `VALIDACIÓN EXITOSA`.